# EDA — Pipeline de Dados de RH

Análise exploratória dos 5 datasets processados.

In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

pd.set_option("display.max_columns", 100)

ibm = pd.read_csv("../data/ibm-attrition/HR-Employee-Attrition-processed.csv")
res = pd.read_csv("../data/hr-resources/HRDataset_v14-processed.csv")
emp = pd.read_csv("../data/hr-employee/HR_Dataset-processed.csv")
co = pd.read_csv("../data/hr-company/Company-processed.csv")
all_in = pd.read_csv("../data/hr-all-in-one/employee-merged-processed.csv")

for name, d in [("IBM", ibm), ("Resources", res), ("Employee", emp), ("Company", co), ("All-in-One", all_in)]:
    print(f"{name}: {d.shape[0]} rows x {d.shape[1]} cols")

IBM: 1470 rows x 44 cols
Resources: 311 rows x 29 cols
Employee: 3400 rows x 31 cols
Company: 4968 rows x 53 cols
All-in-One: 3000 rows x 168 cols


## 1. Visão Geral

In [2]:
summary = []
for name, d in [("IBM Attrition", ibm), ("HR Resources", res), ("HR Employee", emp), ("HR Company", co), ("HR All-in-One", all_in)]:
    summary.append({
        "Dataset": name, "Rows": d.shape[0], "Cols": d.shape[1],
        "Nulls": d.isnull().sum().sum(), "Dupes": d.duplicated().sum(),
    })
pd.DataFrame(summary)

,Dataset,Rows,Cols,Nulls,Dupes
0,IBM Attrition,1470,44,0,0
1,HR Resources,311,29,207,0
2,HR Employee,3400,31,0,0
3,HR Company,4968,53,46522,0
4,HR All-in-One,3000,168,4401,0


## 2. IBM Attrition

In [3]:
rate = ibm["Attrition"].mean() * 100
print(f"Taxa de attrition: {rate:.1f}%")

c = ibm["Attrition"].value_counts()
fig1 = px.bar(x=["Não saiu", "Saiu"], y=[c[0], c[1]],
              color=["Não saiu", "Saiu"], color_discrete_map={"Não saiu": "#51cf66", "Saiu": "#ff6b6b"},
              title="Contagem de Attrition", labels={"x": "", "y": "Count"})
fig1.update_layout(height=350, showlegend=False)
fig1.show()

fig2 = px.pie(values=[c[0], c[1]], names=["Não saiu", "Saiu"],
              color=["Não saiu", "Saiu"], color_discrete_map={"Não saiu": "#51cf66", "Saiu": "#ff6b6b"},
              hole=0.4, title="Proporção")
fig2.update_layout(height=350)
fig2.show()

Taxa de attrition: 16.1%


In [4]:
fig = go.Figure()
for att, color, label in [(0, "#51cf66", "Não saiu"), (1, "#ff6b6b", "Saiu")]:
    s = ibm[ibm["Attrition"] == att]
    fig.add_trace(go.Box(y=s["Age"], name=label, marker_color=color))
fig.update_layout(height=350, title="Idade x Attrition", showlegend=False)
fig.show()

fig = go.Figure()
for att, color, label in [(0, "#51cf66", "Não saiu"), (1, "#ff6b6b", "Saiu")]:
    s = ibm[ibm["Attrition"] == att]
    fig.add_trace(go.Box(y=s["MonthlyIncome"], name=label, marker_color=color))
fig.update_layout(height=350, title="Renda x Attrition", showlegend=False)
fig.show()

In [5]:
job_cols = [c for c in ibm.columns if c.startswith("JobRole_")]
job_attr = {c.replace("JobRole_", ""): ibm[ibm[c] == 1]["Attrition"].mean() * 100 for c in job_cols}
fig = px.bar(x=list(job_attr.keys()), y=list(job_attr.values()),
             color=list(job_attr.values()), color_continuous_scale="Reds",
             title="Attrition por Cargo (%)", labels={"x": "", "y": "%"})
fig.update_layout(height=400, showlegend=False)
fig.show()

In [6]:
corr = ibm.corr(numeric_only=True)["Attrition"].drop("Attrition").abs().sort_values(ascending=False).head(10)
fig = px.bar(x=corr.values, y=corr.index, orientation="h",
             color=corr.values, color_continuous_scale="RdBu_r",
             title="Top 10 Correlações com Attrition", labels={"x": "Correlação", "y": ""})
fig.update_layout(height=400, showlegend=False)
fig.show()

## 3. HR Resources

In [7]:
print(f"Ativos: {(res['HasTerminated'] == 0).sum()}, Demitidos: {(res['HasTerminated'] == 1).sum()}")

fig = px.bar(x=res["EmploymentStatus"].value_counts().index,
             y=res["EmploymentStatus"].value_counts().values,
             color=res["EmploymentStatus"].value_counts().values,
             color_continuous_scale="Viridis", title="Status de Emprego")
fig.update_layout(height=350, showlegend=False)
fig.show()

fig = px.bar(x=["PIP", "Needs Improve", "Fully Meets", "Exceeds"],
             y=res["PerformanceScore"].value_counts().sort_index().values,
             color=["#ff6b6b", "#ffa94d", "#51cf66", "#339af0"],
             title="Performance Score")
fig.update_layout(height=350, showlegend=False)
fig.show()

Ativos: 207, Demitidos: 104


In [8]:
fig = go.Figure()
for t, color, label in [(0, "#51cf66", "Ativo"), (1, "#ff6b6b", "Demitido")]:
    s = res[res["HasTerminated"] == t]
    fig.add_trace(go.Box(y=s["Salary"], name=label, marker_color=color))
fig.update_layout(height=350, title="Salário x Status", showlegend=False)
fig.show()

In [9]:
dept = res.groupby("Department")["HasTerminated"].mean() * 100
fig = px.bar(x=dept.index, y=dept.values, color=dept.values,
             color_continuous_scale="Reds", title="Taxa de Demissão por Dept (%)")
fig.update_layout(height=350, showlegend=False)
fig.show()

## 4. HR Employee

In [10]:
print(f"Attrition: {emp['Attrition'].mean() * 100:.1f}%")

fig = px.bar(x=emp["Gender"].value_counts().index, y=emp["Gender"].value_counts().values,
             color=emp["Gender"].value_counts().values, color_continuous_scale="Viridis",
             title="Distribuição por Gênero")
fig.update_layout(height=350, showlegend=False)
fig.show()

fig = px.bar(x=emp["Attrition"].value_counts().index.map({0: "Não", 1: "Sim"}),
             y=emp["Attrition"].value_counts().values,
             color=["#51cf66", "#ff6b6b"], title="Attrition")
fig.update_layout(height=350, showlegend=False)
fig.show()

Attrition: 50.7%


In [11]:
fig = go.Figure()
fig.add_trace(go.Histogram(x=emp["Age"], nbinsx=30, marker_color="#4dabf7", name="Idade"))
fig.update_layout(height=350, title="Distribuição de Idade")
fig.show()

fig = go.Figure()
fig.add_trace(go.Histogram(x=emp["Monthly_Income"], nbinsx=40, marker_color="#ffa94d", name="Renda"))
fig.update_layout(height=350, title="Distribuição de Renda")
fig.show()

## 5. HR Company

In [12]:
print(f"Ativos: {(co['HasTerminated'] == 0).sum()}, Demitidos: {(co['HasTerminated'] == 1).sum()}")

fig = px.bar(x=co["Gender"].value_counts().index, y=co["Gender"].value_counts().values,
             color=["#f783ac", "#4dabf7"], title="Gênero")
fig.update_layout(height=350, showlegend=False)
fig.show()

fig = px.bar(x=co["Country"].value_counts().index, y=co["Country"].value_counts().values,
             color=co["Country"].value_counts().values, color_continuous_scale="Viridis",
             title="Funcionários por País")
fig.update_layout(height=350, showlegend=False)
fig.show()

Ativos: 2413, Demitidos: 2555


In [13]:
fig = px.box(co, x="Department", y="Salary", color="Department", title="Salário por Departamento")
fig.update_layout(height=400, showlegend=False)
fig.show()

In [14]:
fig = px.scatter(co, x="Age", y="Salary", color="Gender",
                 color_discrete_map={"M": "#4dabf7", "F": "#f783ac"},
                 opacity=0.4, title="Idade x Salário")
fig.update_layout(height=400)
fig.show()

## 6. HR All-in-One

In [ ]:
print(f"Total: {len(all_in)}, Ativos: {(all_in['HasTerminated'] == 0).sum()}, Demitidos: {(all_in['HasTerminated'] == 1).sum()}")

survey_cols = [c for c in all_in.columns if "Score" in c]
fig = go.Figure()
for col in survey_cols:
    fig.add_trace(go.Box(y=all_in[col], name=col.replace(" Score", "")))
fig.update_layout(height=400, title="Scores de Engagement")
fig.show()

In [ ]:
fig = go.Figure()
fig.add_trace(go.Histogram(x=all_in["TenureDays"].dropna(), nbinsx=40, marker_color="#4dabf7"))
fig.update_layout(height=350, title="Tenure (dias)")
fig.show()

## Resumo

| Dataset | Attrition/Demissão | Insights |
|---------|-------------------|----------|
| IBM Attrition | ~16% | Jovens e baixa renda saem mais |
| HR Resources | ~33% | PIP correlaciona com demissão |
| HR Employee | ~15% | Dataset limpo, 4 genders |
| HR Company | ~49% | Multi-país, salários variados |
| HR All-in-One | ~49% | Engagement scores disponíveis |